In [2]:
import pandas as pd
import numpy as np

from pathlib import Path
from collections import defaultdict, deque

BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"

OUTPUT_DIR.mkdir(exist_ok=True)

transactions_df = pd.read_csv(
    DATA_DIR / "transactions.csv"
)

payment_outcomes_df = pd.read_csv(
    DATA_DIR / "payment_outcomes.csv"
)

labels_df = pd.read_csv(
    DATA_DIR / "labels.csv"
)

features_df = pd.read_csv(
    DATA_DIR / "features.csv"
)

print("Feature engineering datasets loaded successfully!")

Feature engineering datasets loaded successfully!


In [3]:
'''
No.	Feature	Business meaning
1	amount_to_avg_ratio	Current amount compared with customer's preceding 90-day average
2	is_new_device	Whether the customer has used this device in the preceding 90 days
3	customer_merchant_distance_km	Distance between customer and merchant
4	is_night	Whether transaction happened between 00:00 and 05:00 IST
5	transaction_count_last_1_hour	Previous transaction count in the last hour
6	transaction_amount_last_24_hours	Previous transaction amount during the last 24 hours
'''

"\nNo.\tFeature\tBusiness meaning\n1\tamount_to_avg_ratio\tCurrent amount compared with customer's preceding 90-day average\n2\tis_new_device\tWhether the customer has used this device in the preceding 90 days\n3\tcustomer_merchant_distance_km\tDistance between customer and merchant\n4\tis_night\tWhether transaction happened between 00:00 and 05:00 IST\n5\ttransaction_count_last_1_hour\tPrevious transaction count in the last hour\n6\ttransaction_amount_last_24_hours\tPrevious transaction amount during the last 24 hours\n"

In [4]:
# Keep the original transactions_df unchanged
source_df = transactions_df[
    ["transaction_id", "customer_id", "transaction_time", "amount_paise"]
].copy()

# Prepare the required columns
source_df["transaction_time"] = pd.to_datetime(
    source_df["transaction_time"],
    format="mixed",
    utc=True
)

source_df["amount_paise"] = pd.to_numeric(
    source_df["amount_paise"], errors="coerce"
)

# Sort transactions customer-wise and chronologically
source_df = source_df.sort_values(
    ["customer_id", "transaction_time"]
)

# Calculate the average amount from the previous 90 days
source_df["previous_90d_avg_paise"] = float("nan")

for customer_id, group in source_df.groupby("customer_id", sort=False):

    rolling_avg = (
        group.rolling(
            "90D",
            on="transaction_time",
            closed="left"
        )["amount_paise"].mean()
    )

    source_df.loc[
        group.index, "previous_90d_avg_paise"
    ] = rolling_avg.to_numpy()

# Calculate amount_to_avg_ratio
source_df["amount_to_avg_ratio"] = (
    source_df["amount_paise"]
    / source_df["previous_90d_avg_paise"].where(
        source_df["previous_90d_avg_paise"] > 0
    )
)

# Keep the computed feature separately
computed_features_df = source_df[
    ["transaction_id", "amount_to_avg_ratio"]
].copy()

In [5]:
computed_features_df.head(10)


,transaction_id,amount_to_avg_ratio
964,TX_0000000965,NaN
965,TX_0000000966,1.173731
7392,TX_0000007393,3.739483
7393,TX_0000007394,6.754269
7394,TX_0000007395,0.801239
8963,TX_0000008964,2.448720
10731,TX_0000010732,0.249916
12102,TX_0000012103,0.112308
12103,TX_0000012104,0.460084
14865,TX_0000014866,0.402535


In [6]:
computed_features_df["amount_to_avg_ratio"].isna().sum()


np.int64(2829)

In [7]:
computed_features_df["amount_to_avg_ratio"].describe()

count    107171.000000
mean          2.771739
std         123.156839
min           0.000066
25%           0.412694
50%           0.746678
75%           1.355559
max       30211.635879
Name: amount_to_avg_ratio, dtype: float64

In [8]:
computed_features_df.loc[
    computed_features_df["amount_to_avg_ratio"].isna(),
    "transaction_id"
].count()

np.int64(2829)

In [10]:
# Select required columns
device_df = transactions_df[
    ["transaction_id", "customer_id", "device_id", "transaction_time"]
].copy()

# Convert transaction time to UTC
device_df["transaction_time"] = pd.to_datetime(
    device_df["transaction_time"],
    format="mixed",
    utc=True
)

# Sort transactions chronologically
device_df = device_df.sort_values(
    ["customer_id", "transaction_time"]
)

# Function to identify a new device
def check_new_device(group):

    results = []
    device_history = {}

    for index, row in group.iterrows():

        current_time = row["transaction_time"]
        current_device = row["device_id"]

        if pd.isna(current_device):
            results.append(float("nan"))
            continue

        # Remove device usage older than 90 days
        cutoff_time = current_time - pd.Timedelta(days=90)

        device_history = {
            device: last_used
            for device, last_used in device_history.items()
            if last_used >= cutoff_time
        }

        # Check whether this device was previously used
        is_new = current_device not in device_history

        results.append(is_new)

        # Update device history after checking current transaction
        device_history[current_device] = current_time

    return pd.Series(results, index=group.index)


# Calculate separately for each customer
device_df["is_new_device"] = (
    device_df.groupby("customer_id", sort=False)
    .apply(check_new_device, include_groups=False)
    .reset_index(level=0, drop=True)
)

# Store the computed feature separately
computed_device_df = device_df[
    ["transaction_id", "is_new_device"]
].copy()

In [11]:
computed_device_df["is_new_device"].value_counts(dropna=False)


is_new_device
False    101557
True       7305
NaN        1138
Name: count, dtype: int64

In [12]:
transactions_df[
    ["customer_location", "merchant_location"]
].head(5)

,customer_location,merchant_location
0,NaN,"{'lat': 12.97, 'lon': 77.59, 'city': 'Bengaluru'}"
1,"{'lat': 19.08, 'lon': 72.88, 'city': 'Mumbai'}","{'lat': 12.97, 'lon': 77.59, 'city': 'Bengaluru'}"
2,"{'lat': 19.08, 'lon': 72.88, 'city': 'Mumbai'}","{'lat': 12.97, 'lon': 77.59, 'city': 'Bengaluru'}"
3,"{'lat': 19.08, 'lon': 72.88, 'city': 'Mumbai'}","{'lat': 12.97, 'lon': 77.59, 'city': 'Bengaluru'}"
4,"{'lat': 19.08, 'lon': 72.88, 'city': 'Mumbai'}","{'lat': 12.97, 'lon': 77.59, 'city': 'Bengaluru'}"


In [13]:
transactions_df["customer_location"].iloc[0]

nan

In [14]:
transactions_df["merchant_location"].iloc[0]

"{'lat': 12.97, 'lon': 77.59, 'city': 'Bengaluru'}"

In [15]:
transactions_df["customer_location"].dropna().head(5)

1    {'lat': 19.08, 'lon': 72.88, 'city': 'Mumbai'}
2    {'lat': 19.08, 'lon': 72.88, 'city': 'Mumbai'}
3    {'lat': 19.08, 'lon': 72.88, 'city': 'Mumbai'}
4    {'lat': 19.08, 'lon': 72.88, 'city': 'Mumbai'}
5    {'lat': 19.08, 'lon': 72.88, 'city': 'Mumbai'}
Name: customer_location, dtype: str

In [16]:
transactions_df["merchant_location"].dropna().head(5)

0    {'lat': 12.97, 'lon': 77.59, 'city': 'Bengaluru'}
1    {'lat': 12.97, 'lon': 77.59, 'city': 'Bengaluru'}
2    {'lat': 12.97, 'lon': 77.59, 'city': 'Bengaluru'}
3    {'lat': 12.97, 'lon': 77.59, 'city': 'Bengaluru'}
4    {'lat': 12.97, 'lon': 77.59, 'city': 'Bengaluru'}
Name: merchant_location, dtype: str

In [17]:
import ast

In [18]:
import numpy as np

def haversine_distance(lat1, lon1, lat2, lon2):

    earth_radius_km = 6371

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    delta_lat = lat2 - lat1
    delta_lon = lon2 - lon1

    a = (
        np.sin(delta_lat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2)
        * np.sin(delta_lon / 2) ** 2
    )

    c = 2 * np.arcsin(np.sqrt(a))

    return earth_radius_km * c

In [19]:
haversine_distance(19.08, 72.88, 12.97, 77.59)

np.float64(845.3775915864525)

In [20]:
# Convert location strings into Python dictionaries
def parse_location(location):

    if pd.isna(location):
        return None

    if isinstance(location, str):
        return ast.literal_eval(location)

    return location


# Calculate distance for each transaction
def calculate_transaction_distance(row):

    customer = parse_location(row["customer_location"])
    merchant = parse_location(row["merchant_location"])

    if customer is None or merchant is None:
        return np.nan

    return haversine_distance(
        customer["lat"],
        customer["lon"],
        merchant["lat"],
        merchant["lon"]
    )


# Create a separate DataFrame
distance_df = transactions_df[
    ["transaction_id", "customer_location", "merchant_location"]
].copy()

distance_df["customer_merchant_distance_km"] = distance_df.apply(
    calculate_transaction_distance,
    axis=1
)

# Keep the computed feature separately
computed_distance_df = distance_df[
    ["transaction_id", "customer_merchant_distance_km"]
].copy()

In [21]:
computed_distance_df["customer_merchant_distance_km"].describe()

count    106743.000000
mean       1046.275056
std         554.756410
min           0.000000
25%         594.531453
50%        1080.743559
75%        1443.099677
max        2428.810609
Name: customer_merchant_distance_km, dtype: float64

In [22]:
# Convert transaction time from UTC to IST
transaction_time_ist = pd.to_datetime(
    transactions_df["transaction_time"],
    format="mixed",
    utc=True
).dt.tz_convert("Asia/Kolkata")

# Create the feature separately
computed_night_df = transactions_df[["transaction_id"]].copy()

computed_night_df["is_night"] = (
    transaction_time_ist.dt.hour < 5
)

In [23]:
computed_night_df["is_night"].value_counts()

is_night
False    94562
True     15438
Name: count, dtype: int64

In [24]:
# Prepare transaction data
velocity_df = transactions_df[
    ["transaction_id", "customer_id", "transaction_time"]
].copy()

velocity_df["transaction_time"] = pd.to_datetime(
    velocity_df["transaction_time"],
    format="mixed",
    utc=True
)

velocity_df = velocity_df.sort_values(
    ["customer_id", "transaction_time"]
)

# Calculate previous transaction count within one hour
velocity_df["transaction_count_last_1_hour"] = 0

for customer_id, group in velocity_df.groupby("customer_id", sort=False):

    counts = (
        group.set_index("transaction_time")["transaction_id"]
        .rolling("1h", closed="left")
        .count()
    )

    velocity_df.loc[
        group.index, "transaction_count_last_1_hour"
    ] = counts.to_numpy()

# Store computed feature separately
computed_velocity_df = velocity_df[
    ["transaction_id", "transaction_count_last_1_hour"]
].copy()

In [25]:
computed_velocity_df["transaction_count_last_1_hour"].describe()

count    46991.000000
mean         2.712030
std          1.974034
min          1.000000
25%          1.000000
50%          2.000000
75%          4.000000
max         10.000000
Name: transaction_count_last_1_hour, dtype: float64

In [26]:
# Prepare transaction data
amount_velocity_df = transactions_df[
    ["transaction_id", "customer_id", "transaction_time", "amount_paise"]
].copy()

amount_velocity_df["transaction_time"] = pd.to_datetime(
    amount_velocity_df["transaction_time"],
    format="mixed",
    utc=True
)

amount_velocity_df = amount_velocity_df.sort_values(
    ["customer_id", "transaction_time"]
)

# Calculate previous 24-hour transaction amount
amount_velocity_df["transaction_amount_last_24_hours"] = 0.0

for customer_id, group in amount_velocity_df.groupby("customer_id", sort=False):

    totals = (
        group.set_index("transaction_time")["amount_paise"]
        .rolling("24h", closed="left")
        .sum()
    )

    amount_velocity_df.loc[
        group.index, "transaction_amount_last_24_hours"
    ] = totals.to_numpy()

# Store separately
computed_amount_velocity_df = amount_velocity_df[
    ["transaction_id", "transaction_amount_last_24_hours"]
].copy()

In [27]:
computed_amount_velocity_df[
    "transaction_amount_last_24_hours"
].describe()

count    5.031000e+04
mean     1.131980e+06
std      1.647209e+07
min      9.620000e+02
25%      1.160882e+05
50%      2.975165e+05
75%      7.448192e+05
max      8.007534e+08
Name: transaction_amount_last_24_hours, dtype: float64

In [28]:
computed_features_df = (
    computed_features_df
    .merge(computed_device_df, on="transaction_id")
    .merge(computed_distance_df, on="transaction_id")
    .merge(computed_night_df, on="transaction_id")
    .merge(computed_velocity_df, on="transaction_id")
    .merge(computed_amount_velocity_df, on="transaction_id")
)

In [29]:
computed_features_df.head()

,transaction_id,amount_to_avg_ratio,is_new_device,customer_merchant_distance_km,is_night,transaction_count_last_1_hour,transaction_amount_last_24_hours
0,TX_0000000965,NaN,True,1144.866173,False,NaN,NaN
1,TX_0000000966,1.173731,False,NaN,False,1.0,364783.0
2,TX_0000007393,3.739483,False,920.081741,False,NaN,NaN
3,TX_0000007394,6.754269,False,920.081741,False,1.0,1482593.0
4,TX_0000007395,0.801239,False,920.081741,False,2.0,6605780.0


In [30]:
computed_features_df.shape

(110000, 7)

In [31]:
reference_features_df = features_df[
    [
        "transaction_id",
        "amount_to_avg_ratio",
        "is_new_device",
        "customer_merchant_distance_km",
        "is_night",
        "txn_count_last_1h",
        "txn_amount_last_24h_paise"
    ]
].copy()

In [33]:
feature_mapping = {
    "amount_to_avg_ratio": (
        "amount_to_avg_ratio_computed",
        "amount_to_avg_ratio_reference"
    ),
    "is_new_device": (
        "is_new_device_computed",
        "is_new_device_reference"
    ),
    "customer_merchant_distance_km": (
        "customer_merchant_distance_km_computed",
        "customer_merchant_distance_km_reference"
    ),
    "is_night": (
        "is_night_computed",
        "is_night_reference"
    ),
    "transaction_count_last_1_hour": (
        "transaction_count_last_1_hour",
        "txn_count_last_1h"
    ),
    "transaction_amount_last_24_hours": (
        "transaction_amount_last_24_hours",
        "txn_amount_last_24h_paise"
    )
}

for feature, (computed_col, reference_col) in feature_mapping.items():

    computed = comparison_df[computed_col]
    reference = comparison_df[reference_col]

    matches = computed.eq(reference) | (
        computed.isna() & reference.isna()
    )

    print(f"{feature}: {matches.mean() * 100:.2f}%")

amount_to_avg_ratio: 2.58%
is_new_device: 100.00%
customer_merchant_distance_km: 8.90%
is_night: 100.00%
transaction_count_last_1_hour: 42.72%
transaction_amount_last_24_hours: 45.74%


In [34]:
features_df["feature_as_of"] = pd.to_datetime(
    features_df["feature_as_of"],
    format="mixed",
    utc=True
)

print(features_df["feature_as_of"].describe())
print(features_df[["feature_as_of"]].head())

count                              110000
mean     2025-07-07 14:42:17.913092+00:00
min      2024-09-30 18:35:15.602000+00:00
25%      2025-03-15 01:43:19.688250+00:00
50%      2025-07-09 02:52:36.484000+00:00
75%      2025-11-02 11:47:43.512250+00:00
max      2026-02-28 18:07:27.471000+00:00
Name: feature_as_of, dtype: object
                  feature_as_of
0 2024-09-30 18:35:15.602+00:00
1 2024-09-30 18:36:10.733+00:00
2 2024-09-30 18:37:47.985+00:00
3 2024-09-30 18:38:19.318+00:00
4 2024-09-30 18:39:15.746+00:00


In [35]:
time_check_df = transactions_df[
    ["transaction_id", "transaction_time"]
].merge(
    features_df[["transaction_id", "feature_as_of"]],
    on="transaction_id"
)

time_check_df["transaction_time"] = pd.to_datetime(
    time_check_df["transaction_time"],
    format="mixed",
    utc=True
)

time_check_df["time_difference"] = (
    time_check_df["feature_as_of"]
    - time_check_df["transaction_time"]
)

print(time_check_df["time_difference"].describe())
print(time_check_df.head())

count             110000
mean     0 days 00:00:00
std      0 days 00:00:00
min      0 days 00:00:00
25%      0 days 00:00:00
50%      0 days 00:00:00
75%      0 days 00:00:00
max      0 days 00:00:00
Name: time_difference, dtype: object
  transaction_id              transaction_time                 feature_as_of  \
0  TX_0000000001 2024-09-30 18:35:15.602+00:00 2024-09-30 18:35:15.602+00:00   
1  TX_0000000002 2024-09-30 18:36:10.733+00:00 2024-09-30 18:36:10.733+00:00   
2  TX_0000000003 2024-09-30 18:37:47.985+00:00 2024-09-30 18:37:47.985+00:00   
3  TX_0000000004 2024-09-30 18:38:19.318+00:00 2024-09-30 18:38:19.318+00:00   
4  TX_0000000005 2024-09-30 18:39:15.746+00:00 2024-09-30 18:39:15.746+00:00   

  time_difference  
0          0 days  
1          0 days  
2          0 days  
3          0 days  
4          0 days  


In [36]:
numeric_features = {
    "amount_to_avg_ratio": (
        "amount_to_avg_ratio_computed",
        "amount_to_avg_ratio_reference"
    ),
    "customer_merchant_distance_km": (
        "customer_merchant_distance_km_computed",
        "customer_merchant_distance_km_reference"
    ),
    "transaction_count_last_1_hour": (
        "transaction_count_last_1_hour",
        "txn_count_last_1h"
    ),
    "transaction_amount_last_24_hours": (
        "transaction_amount_last_24_hours",
        "txn_amount_last_24h_paise"
    )
}

for feature, (computed_col, reference_col) in numeric_features.items():
    computed = comparison_df[computed_col]
    reference = comparison_df[reference_col]

    matches = np.isclose(
        computed,
        reference,
        rtol=1e-5,
        atol=1e-8,
        equal_nan=True
    )

    print(f"{feature}: {matches.mean() * 100:.2f}%")

amount_to_avg_ratio: 24.08%
customer_merchant_distance_km: 98.63%
transaction_count_last_1_hour: 42.72%
transaction_amount_last_24_hours: 45.74%


In [37]:
reason_codes_df = transactions_df[
    ["transaction_id", "account_age"]
].copy()

reason_codes_df = reason_codes_df.merge(
    computed_features_df[
        ["transaction_id", "is_new_device"]
    ],
    on="transaction_id"
)

def assign_reason_codes(row):
    codes = []

    if row["is_new_device"] == True:
        codes.append("R01_NEW_DEVICE")

    if pd.notna(row["account_age"]) and row["account_age"] < 30:
        codes.append("R06_NEW_ACCOUNT")

    return codes


reason_codes_df["reason_codes"] = reason_codes_df.apply(
    assign_reason_codes,
    axis=1
)

print(reason_codes_df["reason_codes"].explode().value_counts())
print(reason_codes_df.head())

reason_codes
R01_NEW_DEVICE     7305
R06_NEW_ACCOUNT    1109
Name: count, dtype: int64
  transaction_id  account_age is_new_device      reason_codes
0  TX_0000000001          836          True  [R01_NEW_DEVICE]
1  TX_0000000002          836         False                []
2  TX_0000000003          836         False                []
3  TX_0000000004          836         False                []
4  TX_0000000005          836         False                []


In [38]:
reason_codes_df = reason_codes_df.merge(
    computed_distance_df[
        ["transaction_id", "customer_merchant_distance_km"]
    ],
    on="transaction_id"
)

reason_codes_df = reason_codes_df.merge(
    computed_night_df[
        ["transaction_id", "is_night"]
    ],
    on="transaction_id"
)

print(reason_codes_df.head())

  transaction_id  account_age is_new_device      reason_codes  \
0  TX_0000000001          836          True  [R01_NEW_DEVICE]   
1  TX_0000000002          836         False                []   
2  TX_0000000003          836         False                []   
3  TX_0000000004          836         False                []   
4  TX_0000000005          836         False                []   

   customer_merchant_distance_km  is_night  
0                            NaN      True  
1                     845.377592      True  
2                     845.377592      True  
3                     845.377592      True  
4                     845.377592      True  


In [39]:
def add_r03(row):
    if row["is_night"] == True:
        row["reason_codes"].append("R03_ODD_HOUR")
    return row["reason_codes"]


reason_codes_df["reason_codes"] = reason_codes_df.apply(
    add_r03,
    axis=1
)

print(reason_codes_df["reason_codes"].explode().value_counts())

reason_codes
R03_ODD_HOUR       15438
R01_NEW_DEVICE      7305
R06_NEW_ACCOUNT     1109
Name: count, dtype: int64


In [40]:
print(
    transactions_df[
        ["customer_location", "merchant_location"]
    ].head(10).to_string(index=False)
)

                                customer_location                                 merchant_location
                                              NaN {'lat': 12.97, 'lon': 77.59, 'city': 'Bengaluru'}
   {'lat': 19.08, 'lon': 72.88, 'city': 'Mumbai'} {'lat': 12.97, 'lon': 77.59, 'city': 'Bengaluru'}
   {'lat': 19.08, 'lon': 72.88, 'city': 'Mumbai'} {'lat': 12.97, 'lon': 77.59, 'city': 'Bengaluru'}
   {'lat': 19.08, 'lon': 72.88, 'city': 'Mumbai'} {'lat': 12.97, 'lon': 77.59, 'city': 'Bengaluru'}
   {'lat': 19.08, 'lon': 72.88, 'city': 'Mumbai'} {'lat': 12.97, 'lon': 77.59, 'city': 'Bengaluru'}
   {'lat': 19.08, 'lon': 72.88, 'city': 'Mumbai'} {'lat': 12.97, 'lon': 77.59, 'city': 'Bengaluru'}
 {'lat': 22.8, 'lon': 86.2, 'city': 'Jamshedpur'}      {'lat': 9.93, 'lon': 76.27, 'city': 'Kochi'}
{'lat': 23.02, 'lon': 72.57, 'city': 'Ahmedabad'}   {'lat': 22.57, 'lon': 88.36, 'city': 'Kolkata'}
{'lat': 23.02, 'lon': 72.57, 'city': 'Ahmedabad'}   {'lat': 22.57, 'lon': 88.36, 'city': 'Kolkata'}


In [41]:
import ast

def get_city(location):
    if pd.isna(location):
        return None

    if isinstance(location, str):
        location = ast.literal_eval(location)

    return location.get("city")


geo_df = transactions_df[
    ["transaction_id", "customer_location", "merchant_location"]
].copy()

geo_df["customer_city"] = geo_df["customer_location"].apply(get_city)
geo_df["merchant_city"] = geo_df["merchant_location"].apply(get_city)

geo_df["city_mismatch"] = (
    geo_df["customer_city"].notna()
    & geo_df["merchant_city"].notna()
    & (geo_df["customer_city"] != geo_df["merchant_city"])
)

print(geo_df["city_mismatch"].value_counts())

city_mismatch
True     100205
False      9795
Name: count, dtype: int64


In [42]:
payment_outcomes_df["status"].value_counts(dropna=False)

status
SUCCESS    97985
FAILED     12015
Name: count, dtype: int64

In [43]:
outcomes_df = payment_outcomes_df[
    ["transaction_id", "outcome_time", "status"]
].copy()

outcomes_df["outcome_time"] = pd.to_datetime(
    outcomes_df["outcome_time"],
    format="mixed",
    utc=True
)

outcomes_df = outcomes_df.sort_values("outcome_time")

print(outcomes_df.head())
print(outcomes_df["outcome_time"].isna().sum())

  transaction_id                  outcome_time   status
0  TX_0000000001 2024-09-30 18:35:16.622+00:00   FAILED
1  TX_0000000002 2024-09-30 18:36:11.876+00:00   FAILED
2  TX_0000000003 2024-09-30 18:37:48.053+00:00  SUCCESS
3  TX_0000000004 2024-09-30 18:38:20.578+00:00   FAILED
4  TX_0000000005 2024-09-30 18:39:16.087+00:00   FAILED
0


In [44]:
failed_outcomes_df = outcomes_df[
    outcomes_df["status"] == "FAILED"
].copy()

failed_attempts_df = transactions_df[
    ["transaction_id", "transaction_time"]
].copy()

failed_attempts_df["transaction_time"] = pd.to_datetime(
    failed_attempts_df["transaction_time"],
    format="mixed",
    utc=True
)

failed_attempts_df["failed_attempts_last_24h"] = 0

failed_times = failed_outcomes_df["outcome_time"].sort_values()

for index, row in failed_attempts_df.iterrows():
    current_time = row["transaction_time"]
    start_time = current_time - pd.Timedelta(hours=24)

    count = (
        (failed_times >= start_time)
        & (failed_times < current_time)
    ).sum()

    failed_attempts_df.loc[
        index, "failed_attempts_last_24h"
    ] = count

print(failed_attempts_df["failed_attempts_last_24h"].value_counts().sort_index())

failed_attempts_last_24h
0      33
1     103
2     488
3     272
4     699
     ... 
62      7
63     22
64     61
65      1
66     23
Name: count, Length: 67, dtype: int64


In [45]:
reason_codes_df = reason_codes_df.merge(
    failed_attempts_df[
        ["transaction_id", "failed_attempts_last_24h"]
    ],
    on="transaction_id"
)

reason_codes_df.loc[
    reason_codes_df["failed_attempts_last_24h"] >= 3,
    "reason_codes"
] = reason_codes_df.loc[
    reason_codes_df["failed_attempts_last_24h"] >= 3,
    "reason_codes"
].apply(lambda codes: codes + ["R04_FAILED_ATTEMPTS"])

print(
    reason_codes_df["reason_codes"]
    .explode()
    .value_counts()
)

reason_codes
R04_FAILED_ATTEMPTS    109376
R03_ODD_HOUR            15438
R01_NEW_DEVICE           7305
R06_NEW_ACCOUNT          1109
Name: count, dtype: int64


In [46]:
failed_outcomes_df = (
    payment_outcomes_df[
        ["transaction_id", "outcome_time", "status"]
    ]
    .merge(
        transactions_df[["transaction_id", "customer_id"]],
        on="transaction_id",
        how="left"
    )
)

failed_outcomes_df = failed_outcomes_df[
    failed_outcomes_df["status"] == "FAILED"
].copy()

print(failed_outcomes_df.head())

  transaction_id                   outcome_time  status         customer_id
0  TX_0000000001  2024-09-30T18:35:16.622+00:00  FAILED  C_b90ff711ce3e3dba
1  TX_0000000002  2024-09-30T18:36:11.876+00:00  FAILED  C_b90ff711ce3e3dba
3  TX_0000000004  2024-09-30T18:38:20.578+00:00  FAILED  C_b90ff711ce3e3dba
4  TX_0000000005  2024-09-30T18:39:16.087+00:00  FAILED  C_b90ff711ce3e3dba
7  TX_0000000008  2024-09-30T19:59:15.275+00:00  FAILED  C_9656833c6d36f7c7


In [47]:
failed_outcomes_df["outcome_time"] = pd.to_datetime(
    failed_outcomes_df["outcome_time"],
    format="mixed",
    utc=True
)

failed_attempts_df = transactions_df[
    ["transaction_id", "customer_id", "transaction_time"]
].copy()

failed_attempts_df["transaction_time"] = pd.to_datetime(
    failed_attempts_df["transaction_time"],
    format="mixed",
    utc=True
)

failed_attempts_df["failed_attempts_last_24h"] = 0

for customer_id, group in failed_attempts_df.groupby("customer_id"):
    customer_failures = failed_outcomes_df.loc[
        failed_outcomes_df["customer_id"] == customer_id,
        "outcome_time"
    ].sort_values()

    for index, row in group.iterrows():
        current_time = row["transaction_time"]
        start_time = current_time - pd.Timedelta(hours=24)

        count = (
            (customer_failures >= start_time)
            & (customer_failures < current_time)
        ).sum()

        failed_attempts_df.loc[
            index, "failed_attempts_last_24h"
        ] = count

print(
    failed_attempts_df["failed_attempts_last_24h"]
    .value_counts()
    .sort_index()
)

failed_attempts_last_24h
0    93239
1     3852
2     3032
3     3806
4     5883
5      167
6        7
7        5
8        9
Name: count, dtype: int64


In [48]:
reason_codes_df["reason_codes"] = reason_codes_df["reason_codes"].apply(
    lambda codes: [
        code for code in codes
        if code != "R04_FAILED_ATTEMPTS"
    ]
)

reason_codes_df = reason_codes_df.drop(
    columns=["failed_attempts_last_24h"],
    errors="ignore"
)

reason_codes_df = reason_codes_df.merge(
    failed_attempts_df[
        ["transaction_id", "failed_attempts_last_24h"]
    ],
    on="transaction_id"
)

reason_codes_df.loc[
    reason_codes_df["failed_attempts_last_24h"] >= 3,
    "reason_codes"
] = reason_codes_df.loc[
    reason_codes_df["failed_attempts_last_24h"] >= 3,
    "reason_codes"
].apply(lambda codes: codes + ["R04_FAILED_ATTEMPTS"])

In [49]:
reason_codes_df["reason_codes"].explode().value_counts()

reason_codes
R03_ODD_HOUR           15438
R04_FAILED_ATTEMPTS     9877
R01_NEW_DEVICE          7305
R06_NEW_ACCOUNT         1109
Name: count, dtype: int64

In [51]:
%whos

Variable                         Type           Data/Info
---------------------------------------------------------
BASE_DIR                         WindowsPath    d:\EURON_TASK\sentinel-feature-pipeline
DATA_DIR                         WindowsPath    d:\EURON_TASK\sentinel-feature-pipeline\data
OUTPUT_DIR                       WindowsPath    d:\EURON_TASK\sentinel-feature-pipeline\output
Path                             type           <class 'pathlib.Path'>
add_r03                          function       <function add_r03 at 0x0000025252CB2E50>
amount_velocity_df               DataFrame      Shape: (110000, 5)
assign_reason_codes              function       <function assign_reason_c<...>es at 0x00000252546EF110>
ast                              module         <module 'ast' from 'c:\\U<...>86_64-none\\Lib\\ast.py'>
calculate_transaction_distance   function       <function calculate_trans<...>ce at 0x00000252522ECB40>
check_new_device                 function       <function check_new_d

In [52]:
print(transactions_df["merchant_category"].value_counts())

merchant_category
grocery           20358
quick_commerce    16606
food              14902
electronics       14206
gaming            10692
gift_cards         7704
healthcare         7219
travel             6398
education          6288
other              5627
Name: count, dtype: int64


In [54]:
list((DATA_DIR / "tables").rglob("*threshold*"))

[]

In [55]:
%whos

Variable                         Type           Data/Info
---------------------------------------------------------
BASE_DIR                         WindowsPath    d:\EURON_TASK\sentinel-feature-pipeline
DATA_DIR                         WindowsPath    d:\EURON_TASK\sentinel-feature-pipeline\data
OUTPUT_DIR                       WindowsPath    d:\EURON_TASK\sentinel-feature-pipeline\output
Path                             type           <class 'pathlib.Path'>
add_r03                          function       <function add_r03 at 0x0000025252CB2E50>
amount_velocity_df               DataFrame      Shape: (110000, 5)
assign_reason_codes              function       <function assign_reason_c<...>es at 0x00000252546EF110>
ast                              module         <module 'ast' from 'c:\\U<...>86_64-none\\Lib\\ast.py'>
calculate_transaction_distance   function       <function calculate_trans<...>ce at 0x00000252522ECB40>
check_new_device                 function       <function check_new_d

In [56]:
for file in DATA_DIR.rglob("*.csv*"):
    print(file.relative_to(DATA_DIR))

customers.csv
devices.csv
features.csv
labels.csv
merchants.csv
payment_outcomes.csv
transactions.csv


In [57]:
features_df["amount_to_avg_ratio"].describe()

count    107171.000000
mean          2.771739
std         123.156839
min           0.000100
25%           0.412700
50%           0.746700
75%           1.355550
max       30211.635900
Name: amount_to_avg_ratio, dtype: float64

In [58]:
[
    (name, type(obj).__name__)
    for name, obj in globals().items()
    if callable(getattr(obj, "table", None))
]

[]

In [59]:
final_reason_codes_df = reason_codes_df[
    ["transaction_id", "reason_codes"]
].copy()

final_reason_codes_df["reason_codes"] = final_reason_codes_df[
    "reason_codes"
].apply(lambda codes: ", ".join(codes) if codes else "NO_REASON_CODE")

final_reason_codes_df.to_csv(
    "sentinel_reason_codes_output.csv",
    index=False
)

print("Final reason-code output generated successfully!")
print("Total transactions:", len(final_reason_codes_df))
print("Output file: sentinel_reason_codes_output.csv")
print("\nReason-code distribution:")
print(
    reason_codes_df["reason_codes"]
    .explode()
    .value_counts()
)

Final reason-code output generated successfully!
Total transactions: 110000
Output file: sentinel_reason_codes_output.csv

Reason-code distribution:
reason_codes
R03_ODD_HOUR           15438
R04_FAILED_ATTEMPTS     9877
R01_NEW_DEVICE          7305
R06_NEW_ACCOUNT         1109
Name: count, dtype: int64
